# 04 · Nettoyage : appliquer les règles validées

**Projet 4 : Analyse des données de vente et retours** · Sprint 2 · Livrable L3 (nettoyage)

**Objectif** : appliquer aux ventes jointes les règles décidées par l'équipe après le diagnostic (`03_diagnostic`), garder une trace de chaque modification, puis écrire la table propre dans `data/clean`.

| Étape | Ce qu'on fait | Décision de l'équipe |
|---|---|---|
| 1. Chargement | Lire les ventes jointes et garder une copie de référence | |
| 2. Règle N1 | Retirer les espaces en trop dans les colonnes texte | Correction évidente |
| 3. Règle N2 | Supprimer `continent` et `date_fermeture` | Une seule valeur, ou entièrement vide |
| 4. Règle N3 | Ajouter l'indicateur `promo_hors_periode` | On garde les ventes et leurs remises |
| 5. Règle N4 | Ajouter l'indicateur `grande_quantite` | On garde les grandes quantités |
| 6. Règle N5 | Harmoniser les types techniques | Aucune valeur ne change |
| 7. Règle N6 | Limiter la quantité remisée à la quantité vendue | Seule valeur modifiée : une quantité, sur 2 331 lignes. Aucun montant ne change |
| 8. Règle N7 | Imputer l'effectif manquant d'un magasin | Médiane des magasins comparables, indicateur `effectif_impute` |
| 9. Règle N8 | Ajouter l'indicateur `avant_ouverture` | On garde les 189 ventes datées avant l'ouverture du magasin |
| 10. Dimensions | Appliquer les règles N1 et N7 aux tables de dimension | |
| 11. Validation | Vérifier que rien d'autre n'a changé | |
| 12. Enregistrement | Écrire la table propre, le journal, les corrections et le rapport de validation | |

**Lit dans** : `data/interim`
**Périmètre** : les ventes complètes de nos 56 boutiques sur trois années (8 catégories, 12 trimestres : 2019, année de référence, puis 2020 et 2021), voir `00_inventaire`, section 9.

**Écrit dans** : `data/clean` (`fait_ventes.parquet`, `dimensions/`, `journal_nettoyage.csv`, `corrections_remise.csv`, `rapport_validation.csv`)

**Ce que ce notebook ne fait PAS, par décision de l'équipe**

- Il ne modifie **aucun montant**. Le montant des ventes reste tel quel : c'est le chiffre d'affaires brut, égal au cent près à celui du plan stratégique de Contoso.
- Il ne modifie **aucune quantité, à une exception près** : la règle N6, qui ramène la quantité remisée à la quantité vendue sur 2 331 lignes (0,33 %).
- Il ne modifie pas la colonne `cout_total`, même si elle est calculée de trois façons selon les lignes. Les coûts homogènes (brut, retour, net) sont recalculés dans `05_variables`.
- Il ne corrige pas le mécanisme des promotions : les montants de remise enregistrés sont gardés tels quels. Le taux de remise effectif est calculé dans `05_variables`.
- Il ne supprime **aucune ligne**. Il n'impute qu'**une seule valeur** : l'effectif manquant d'un magasin (règle N7), signalé par un indicateur.

## 0. Préparation

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config -> chemins des dossiers (INTERIM, DIM_REALIGNEES, CLEAN, CLEAN_DIMENSIONS)
from src import config

config.creer_dossiers()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)
print("Racine du projet :", config.ROOT)

# ---------------------------------------------------------------------------
# 3. Journal de nettoyage
# ---------------------------------------------------------------------------
# Chaque règle appliquée ajoute une ou plusieurs entrées : on peut ainsi dire, pour chaque
# modification, ce qui a été fait, sur combien de lignes, et pourquoi.
journal = []


def journaliser(regle, cible, action, nb_lignes, justification):
    """Ajoute une entrée au journal de nettoyage et l'affiche.

    regle         : code de la règle (N1, N2...)
    cible         : colonne ou table concernée
    action        : ce qui a été fait
    nb_lignes     : nombre de lignes modifiées ou marquées
    justification : pourquoi, en référence au diagnostic
    """
    nb_lignes = int(nb_lignes)
    journal.append({"regle": regle, "cible": cible, "action": action,
                    "nb_lignes": nb_lignes, "justification": justification})
    nb_lisible = f"{nb_lignes:,}".replace(",", " ")   # séparateur de milliers : espace
    print(f"[{regle}] {cible} : {action} ({nb_lisible} lignes)")


def nettoyer_texte(serie):
    """Retire les espaces de début et de fin et réduit les espaces multiples à un seul.

    Exemples : "Litware " -> "Litware" ; "Contoso Moscow  No.1 Store" -> "Contoso Moscow No.1 Store".
    Les valeurs manquantes restent manquantes.
    """
    return serie.str.strip().str.replace(r"\s+", " ", regex=True)

## 1. Chargement

In [ ]:
chemin = config.INTERIM / "ventes_jointes.parquet"
if not chemin.exists():
    raise FileNotFoundError(f"{chemin} introuvable : exécuter d'abord 01_etl puis 02_flocon.")

ventes = pd.read_parquet(chemin)

# Copie de référence, jamais modifiée : elle sert à la validation (section 11) pour prouver
# que les mesures n'ont pas changé.
reference = ventes.copy()

print(f"Lignes   : {len(ventes):,}".replace(",", " "))
print(f"Colonnes : {ventes.shape[1]}")
print(f"Chiffre d'affaires : {float(ventes.montant_ventes.sum()):,.2f} $".replace(",", " "))

## 2. Règle N1 : espaces en trop dans les colonnes texte

**Constat du diagnostic (section 5)** : cinq colonnes contiennent des espaces en trop (`categorie`, `sous_categorie`, `nom_boutique`, `marque`, `nom_produit`). Dans `marque`, `"Litware "` et `"Litware"` sont comptées comme deux marques différentes.

**Règle** : on retire les espaces de début et de fin, et on réduit les doubles espaces. La règle est appliquée à **toutes** les colonnes texte, pas seulement aux cinq repérées : si une nouvelle anomalie apparaît dans un prochain fichier, elle sera traitée aussi.

In [ ]:
# Colonnes texte de la table (le type s'appelle "object" ou "str" selon la version de pandas)
colonnes_texte = [c for c in ventes.columns if ventes[c].dtype == object or str(ventes[c].dtype) in ("str", "string")]
print("Colonnes texte :", colonnes_texte, "\n")

for col in colonnes_texte:
    avant = ventes[col]
    apres = nettoyer_texte(avant)
    # Lignes réellement modifiées (on ne compte pas les valeurs manquantes)
    modifiees = ((avant != apres) & avant.notna()).sum()
    if modifiees:
        modalites_avant, modalites_apres = avant.nunique(), apres.nunique()
        ventes[col] = apres
        journaliser("N1", col,
                    f"espaces en trop retirés ; modalités : {modalites_avant} -> {modalites_apres}",
                    modifiees, "Diagnostic section 5 : une même modalité écrite de deux façons")

# Vérification : il ne doit plus rester aucun espace en trop
for col in colonnes_texte:
    restant = ((ventes[col] != nettoyer_texte(ventes[col])) & ventes[col].notna()).sum()
    assert restant == 0, f"Il reste des espaces en trop dans {col}"
print("\nPlus aucun espace en trop dans les colonnes texte.")
print("Marques après nettoyage :", sorted(ventes.marque.unique()))

## 3. Règle N2 : colonnes sans information

**Constat du diagnostic (section 5)** :

- `continent` vaut « Europe » sur toutes les lignes, parce que toutes les boutiques de nos ventes sont européennes ;
- `date_fermeture` est vide sur toutes les lignes, parce qu'aucune de nos boutiques n'a fermé.

**Règle** : on supprime ces deux colonnes de la table propre. Ce n'est pas une correction, c'est du rangement : une colonne qui a la même valeur partout ne permet aucune comparaison. L'information n'est pas perdue, elle reste dans `data/interim` et dans la dimension boutique.

Avant de supprimer, le code **revérifie** que chaque colonne est bien constante ou vide. Si un nouveau fichier contenait une boutique hors d'Europe ou une boutique fermée, la colonne serait conservée.

In [ ]:
COLONNES_SANS_INFORMATION = ["continent", "date_fermeture"]

for col in COLONNES_SANS_INFORMATION:
    if col not in ventes.columns:
        print(f"{col} : déjà absente de la table")
        continue
    nb_valeurs = ventes[col].nunique(dropna=True)   # nombre de valeurs différentes, hors vides
    if nb_valeurs <= 1:
        valeur = ventes[col].dropna().unique()
        description = f"valeur unique : {valeur[0]}" if len(valeur) else "entièrement vide"
        ventes = ventes.drop(columns=col)
        journaliser("N2", col, f"colonne supprimée ({description})", len(ventes),
                    "Diagnostic section 5 : colonne constante ou vide, sans utilité pour l'analyse")
    else:
        print(f"{col} : {nb_valeurs} valeurs différentes -> colonne CONSERVÉE (la règle ne s'applique plus)")

## 4. Règle N3 : promotions hors période

**Constat du diagnostic (section 8)** : 29 712 ventes (4,15 %) sont rattachées à une promotion *European Holiday* qui ne commence qu'en octobre de la même année. Elles sont toutes datées de janvier : 28 454 en janvier 2019 (promotion 10) et 1 258 en janvier 2021 (promotion 28). La campagne court d'octobre à fin janvier : ce sont les ventes de fin de campagne, rattachées à l'édition suivante. Pour janvier 2021, la promotion jumelle (la 19 : même nom, même taux) couvre bien ces dates. Pour janvier 2019, l'édition précédente n'existe pas dans la table des promotions, qui commence en 2019.

**Règle** : on garde ces ventes et leurs remises, qui sont réelles. On ne change pas l'identifiant de promotion (on ne peut pas prouver l'erreur, et pour 2019 il n'existe pas d'identifiant correct). On ajoute un indicateur `promo_hors_periode` pour pouvoir, au choix, inclure ou exclure ces lignes d'une analyse par promotion.

In [ ]:
# Vrai si la vente est datée avant le début ou après la fin de sa promotion
ventes["promo_hors_periode"] = (ventes.date < ventes.promo_date_debut) | (ventes.date > ventes.promo_date_fin)

journaliser("N3", "promo_hors_periode", "indicateur ajouté (vente datée hors de la période de sa promotion)",
            ventes.promo_hors_periode.sum(),
            "Diagnostic section 8 : ventes gardées, identifiant de promotion non modifié")

display(ventes[ventes.promo_hors_periode]
        .groupby(["id_promotion", "nom_promotion", "promo_date_debut", "promo_date_fin"])
        .agg(nb_ventes=("id_vente", "size"), premiere_vente=("date", "min"), derniere_vente=("date", "max"))
        .reset_index())

## 5. Règle N4 : grandes quantités

**Constat du diagnostic (section 10)** : 14,4 % des lignes dépassent la borne de l'écart interquartile (18 unités), jusqu'à 2 880 unités. La part est plus forte qu'avec deux années, parce que 2019 abaisse la quantité habituelle (médiane de 8 unités par ligne en 2019, contre 10 en 2020 et 12 en 2021) et donc la borne. Elles respectent toutes la règle du montant et concernent de petits accessoires vendus en ligne ou à des revendeurs : ce sont de grosses ventes cohérentes, pas des erreurs.

**Règle** : on les garde. On ajoute une colonne `grande_quantite`, qui vaut vrai ou faux. Elle ne modifie et ne supprime rien : c'est une étiquette.

**Comment le seuil est calculé (méthode de l'écart interquartile, IQR)**

1. On trie les quantités vendues et on repère deux valeurs : **Q1**, sous laquelle se trouvent 25 % des lignes, et **Q3**, sous laquelle se trouvent 75 % des lignes.
2. L'écart **IQR = Q3 − Q1** mesure l'étendue de la moitié centrale des lignes, c'est-à-dire ce qu'est une quantité « habituelle ».
3. La **borne haute = Q3 + 1,5 × IQR**. Au-delà, une valeur est dite extrême : elle est loin de ce qu'on observe d'habitude.
4. Toute ligne dont la quantité dépasse la borne haute reçoit `grande_quantite = vrai`.

Le coefficient 1,5 est la convention statistique habituelle (c'est celle des boîtes à moustaches). Le seuil n'est pas choisi à la main : il est recalculé à partir des données, donc il s'adapte si de nouveaux fichiers arrivent.

**Pourquoi marquer plutôt que supprimer**

- Ces ventes sont réelles et cohérentes : les supprimer fausserait le chiffre d'affaires.
- Mais elles pèsent très lourd : quelques lignes peuvent à elles seules tirer une moyenne ou une corrélation.
- Avec l'étiquette, on peut refaire n'importe quelle analyse **avec** et **sans** ces lignes. Si la conclusion est la même dans les deux cas, elle est solide. Si elle change, c'est qu'elle reposait sur quelques très grosses ventes, et il faut le dire.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Calcul du seuil, étape par étape
# ---------------------------------------------------------------------------
q1, q3 = ventes.quantite_vente.quantile([0.25, 0.75])   # 25 % et 75 % des lignes sont en dessous
iqr = q3 - q1                                           # étendue de la moitié centrale
borne_haute = q3 + 1.5 * iqr                            # au-delà : quantité extrême
print(f"Q1 (25 % des lignes en dessous)  = {q1:.0f} unités")
print(f"Q3 (75 % des lignes en dessous)  = {q3:.0f} unités")
print(f"IQR = Q3 − Q1                    = {iqr:.0f} unités")
print(f"Borne haute = Q3 + 1,5 × IQR     = {q3:.0f} + 1,5 × {iqr:.0f} = {borne_haute:.1f} unités")
# Il n'y a pas de borne basse utile ici : Q1 − 1,5 × IQR est négatif, et aucune quantité n'est négative.

# ---------------------------------------------------------------------------
# 2. Étiquette : vrai si la quantité vendue dépasse la borne haute
# ---------------------------------------------------------------------------

ventes["grande_quantite"] = ventes.quantite_vente > borne_haute

journaliser("N4", "grande_quantite", f"indicateur ajouté (quantité vendue > {borne_haute:.1f}, borne IQR)",
            ventes.grande_quantite.sum(),
            "Diagnostic section 10 : valeurs extrêmes mais cohérentes, gardées")

# ---------------------------------------------------------------------------
# 3. Poids de ces lignes : c'est ce qui justifie de les étiqueter
# ---------------------------------------------------------------------------
poids = ventes.groupby("grande_quantite").agg(lignes=("id_vente", "size"),
                                              unites_vendues=("quantite_vente", "sum"),
                                              chiffre_affaires=("montant_ventes", "sum"),
                                              quantite_mediane=("quantite_vente", "median"),
                                              prix_unitaire_median=("prix_unitaire", "median"))
for col in ["lignes", "unites_vendues", "chiffre_affaires"]:
    poids[f"part_{col}"] = poids[col] / poids[col].sum()
poids.index = poids.index.map({False: "quantité habituelle", True: "grande quantité"})
display(poids.style.format({"lignes": "{:,.0f}", "unites_vendues": "{:,.0f}", "chiffre_affaires": "{:,.0f}",
                            "quantite_mediane": "{:.0f}", "prix_unitaire_median": "{:,.2f}",
                            "part_lignes": "{:.1%}", "part_unites_vendues": "{:.1%}", "part_chiffre_affaires": "{:.1%}"}))
# Lecture : peu de lignes, mais une grande part des unités vendues. Le prix unitaire médian de ces
# lignes est bas : ce sont surtout de petits accessoires vendus en volume.

# Répartition par canal : où se trouvent ces grandes quantités ?
display(pd.crosstab(ventes.canal, ventes.grande_quantite.map({False: "quantité habituelle", True: "grande quantité"}),
                    normalize="index").style.format("{:.1%}"))

## 6. Règle N5 : types techniques

Certaines colonnes ont un type dit « nullable » (`Int64`, `Float64`, avec une majuscule), hérité des étapes précédentes. Comme elles ne contiennent aucun vide, on les convertit vers les types classiques (`int64`, `float64`), mieux acceptés par les bibliothèques d'analyse et de modélisation. **Aucune valeur ne change.**

In [ ]:
for col in ventes.columns:
    type_avant = str(ventes[col].dtype)
    if type_avant in ("Int64", "Int32", "Float64") and ventes[col].notna().all():
        type_apres = type_avant.lower()
        ventes[col] = ventes[col].astype(type_apres)
        journaliser("N5", col, f"type {type_avant} -> {type_apres} (aucune valeur modifiée)", 0,
                    "Compatibilité avec numpy, scikit-learn et Power BI")

## 7. Règle N6 : remise sur plus d'unités que la vente

**Constat du diagnostic (section 7, règle R5)** : sur 2 331 lignes, la quantité remisée dépasse la quantité vendue (par exemple 2 unités vendues, 4 remisées). Elles sont dans deux sous-catégories : *Download Games* (1 474 lignes de 2019) et *Washers & Dryers* (471 lignes de 2019, 386 de 2021). Le coût et le chiffre d'affaires portent bien sur la quantité vendue : seule la remise est calculée sur trop d'unités.

**Comment on a tranché.** Le diagnostic a testé trois explications (unités offertes, pack, erreur sur la quantité remisée) et comparé avec les mêmes promotions dans les autres boutiques. Seule l'erreur est cohérente avec les montants : la quantité remisée est tirée entre 0 et 4 sans tenir compte de la quantité vendue, et dépasse quand la vente fait 2 ou 3 unités. Le fichier étant généré, il n'y a pas de mécanisme commercial à retrouver.

**Règle** : on ne peut pas remiser plus d'unités qu'on n'en vend. Sur ces lignes, la **quantité remisée** est ramenée à la quantité vendue. C'est la seule valeur modifiée.

**Les montants ne changent pas.** Le montant de la remise et le montant des ventes restent ceux de la source : la remise enregistrée s'applique simplement aux unités vendues. Exemple : 2 unités vendues, 4 remisées, remise de 2 122,32 $ devient 2 unités vendues, 2 remisées, remise de 2 122,32 $.

**Pourquoi ne pas recalculer la remise** : cela augmenterait le montant des ventes d'autant (règle R1 de l'énoncé : ventes = prix × quantité − remise), alors que le chiffre d'affaires de ces lignes est cohérent avec la quantité vendue et que nos totaux sont égaux à ceux du plan stratégique de Contoso au cent près (`00_inventaire`, section 9). On garde donc cet accord, indispensable pour rapprocher les ventes des charges.

**Traçabilité** : un indicateur `remise_corrigee` marque les lignes, et la quantité d'origine de chaque ligne est enregistrée dans `corrections_remise.csv`.

**Conséquence à connaître** : sur ces lignes, la remise conservée ne correspond plus au taux annoncé appliqué à la quantité remisée (règle R2). Leur taux de remise effectif dépasse le taux annoncé, jusqu'à 40 % pour une promotion de 20 %. L'indicateur permet de les isoler dans les analyses de remise.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Repérer les lignes : quantité remisée supérieure à la quantité vendue
# ---------------------------------------------------------------------------
a_corriger = ventes.quantite_remise > ventes.quantite_vente
ventes["remise_corrigee"] = a_corriger

# ---------------------------------------------------------------------------
# 2. Garder la quantité d'origine de ces lignes AVANT de la modifier
# ---------------------------------------------------------------------------
# Une seule colonne est modifiée par la règle N6. Les montants sont listés ici pour mémoire,
# afin que la table des corrections montre qu'ils n'ont pas bougé.
COLONNES_N6 = ["quantite_remise"]
corrections = ventes.loc[a_corriger, ["id_vente", "date", "id_boutique", "id_produit", "id_promotion",
                                      "quantite_vente", "prix_unitaire", "taux_remise_annonce",
                                      "quantite_remise", "montant_remise", "montant_ventes"]].copy()
corrections = corrections.rename(columns={"quantite_remise": "quantite_remise_origine"})

# ---------------------------------------------------------------------------
# 3. Appliquer la règle : la quantité remisée ne peut pas dépasser la quantité vendue
# ---------------------------------------------------------------------------
# Le montant de la remise et le montant des ventes ne sont PAS recalculés : la remise
# enregistrée s'applique aux unités vendues, et le chiffre d'affaires reste celui de la source.
ventes.loc[a_corriger, "quantite_remise"] = ventes.loc[a_corriger, "quantite_vente"]

# ---------------------------------------------------------------------------
# 4. Compléter la table des corrections
# ---------------------------------------------------------------------------
corrections.insert(corrections.columns.get_loc("quantite_remise_origine") + 1,
                   "quantite_remise_corrige", ventes.loc[a_corriger, "quantite_remise"].values)
# Taux de remise effectif de la ligne (remise ÷ montant avant remise) : il dépasse le taux annoncé
corrections["taux_remise_effectif"] = corrections.montant_remise / (corrections.prix_unitaire * corrections.quantite_vente)

unites_retirees = int((corrections.quantite_remise_origine - corrections.quantite_remise_corrige).sum())
print(f"Lignes corrigées                    : {int(a_corriger.sum())}")
print(f"Unités remisées retirées            : {unites_retirees}")
print(f"Montant des remises (inchangé)      : {float(ventes.montant_remise.sum()):,.2f} $".replace(",", " "))
print(f"Chiffre d'affaires (inchangé)       : {float(ventes.montant_ventes.sum()):,.2f} $".replace(",", " "))
print(f"Taux de remise effectif de ces lignes : de {corrections.taux_remise_effectif.min():.1%} "
      f"à {corrections.taux_remise_effectif.max():.1%}")

journaliser("N6", "quantite_remise", "quantité remisée ramenée à la quantité vendue (montants inchangés)",
            int(a_corriger.sum()),
            "Diagnostic section 7 (R5) : remise sur plus d'unités que la vente ; quantités d'origine dans corrections_remise.csv")
journaliser("N6", "remise_corrigee", "indicateur ajouté (ligne dont la quantité remisée a été limitée)", int(a_corriger.sum()),
            "Permet d'inclure ou d'exclure ces lignes d'une analyse des remises")

print("\nExemples (quantité d'origine et corrigée, montants inchangés) :")
display(corrections[["id_vente", "quantite_vente", "prix_unitaire", "taux_remise_annonce",
                     "quantite_remise_origine", "quantite_remise_corrige",
                     "montant_remise", "montant_ventes", "taux_remise_effectif"]].head(5))

## 8. Règle N7 : effectif manquant d'un magasin

**Constat du diagnostic (section 3)** : la seule valeur manquante de la table est l'effectif d'un magasin, *Contoso Cheshire Store* (Royaume-Uni, 680 m²). Toutes les autres colonnes sont complètes, sauf `nom_ferie`, vide par construction les jours non fériés.

**Pourquoi la traiter.** Jusqu'ici ce vide était toléré, parce que l'effectif ne servait pas aux questions Q1 à Q5. Il devient gênant pour la suite : la typologie des magasins et toute analyse des moyens par magasin ont besoin d'une valeur pour chacun. Un magasin sans effectif serait écarté sans qu'on le voie.

**Les options envisagées**

| Option | Verdict | Raison |
|---|---|---|
| Laisser vide | Écartée | Le magasin disparaît de toute analyse qui utilise l'effectif |
| Supprimer le magasin | Écartée | Ses ventes et ses charges sont complètes et justes |
| Déduire l'effectif de la masse salariale | Écartée | Dans les magasins, les charges de personnel ne suivent pas l'effectif (`05b_charges`, sections 4 et 7), parce qu'elles sont calées sur les ventes de la première année puis reconduites : la corrélation de rang est nulle (−0,08) et le coût par employé va de 5 100 $ à 50 000 $ par an. Testée sur les 53 magasins dont l'effectif est connu, cette déduction se trompe de plus de 25 % pour près de trois magasins sur quatre et ne fait pas mieux que la médiane (erreur moyenne de 55 % contre 46 %). Pour ce magasin, elle donnerait d'ailleurs presque la même valeur |
| **Médiane des magasins comparables** | **Retenue** | Valeur centrale, insensible aux magasins à très gros effectif. C'est le choix habituel quand aucune variable n'explique la valeur manquante |

**Règle** : l'effectif manquant d'un magasin est remplacé par la médiane de l'effectif des magasins du même pays et de même surface. S'il y a moins de 5 magasins comparables, on prend la médiane des magasins du même pays. Un indicateur `effectif_impute` marque les lignes concernées.

**Limite assumée.** L'effectif n'est lié ni à la masse salariale, ni à la surface, ni aux ventes : aucune méthode ne peut retrouver la vraie valeur. La valeur imputée est plausible, pas exacte. L'indicateur permet d'exclure ce magasin d'un calcul (par exemple un coût par employé) ou de vérifier qu'une conclusion ne dépend pas de lui.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Une ligne par boutique : repérer celles dont l'effectif est vide
# ---------------------------------------------------------------------------
boutiques = ventes.drop_duplicates("id_boutique")[["id_boutique", "nom_boutique", "type_boutique", "pays", "surface", "effectif"]]
sans_effectif = boutiques[boutiques.effectif.isna()]
print(f"Boutiques sans effectif : {len(sans_effectif)} sur {len(boutiques)}")
display(sans_effectif)

# ---------------------------------------------------------------------------
# 2. Valeur imputée : médiane des magasins comparables
# ---------------------------------------------------------------------------
MIN_COMPARABLES = 5          # en dessous, le groupe est trop petit : on élargit au pays
ventes["effectif_impute"] = ventes.effectif.isna()
effectifs_imputes = {}       # id_boutique -> valeur imputée, repris pour la dimension des boutiques (section 10)
for _, b in sans_effectif.iterrows():
    connus = boutiques[(boutiques.type_boutique == b.type_boutique) & boutiques.effectif.notna()]
    comparables = connus[(connus.pays == b.pays) & (connus.surface == b.surface)]
    critere = f"même pays ({b.pays}) et même surface ({b.surface:.0f} m²)"
    if len(comparables) < MIN_COMPARABLES:
        comparables = connus[connus.pays == b.pays]
        critere = f"même pays ({b.pays})"
    valeur = float(comparables.effectif.median())
    effectifs_imputes[int(b.id_boutique)] = valeur
    lignes_touchees = int((ventes.id_boutique == b.id_boutique).sum())
    ventes.loc[ventes.id_boutique == b.id_boutique, "effectif"] = valeur
    print(f"{b.nom_boutique} : {len(comparables)} magasins comparables, {critere}")
    print(f"   effectifs comparables : {sorted(comparables.effectif.astype(int))}")
    print(f"   valeur imputée (médiane) : {valeur:.0f} · moyenne pour comparaison : {comparables.effectif.mean():.1f}")
    journaliser("N7", "effectif", f"valeur manquante imputée : {valeur:.0f} (médiane de {len(comparables)} magasins, {critere})",
                lignes_touchees, "Diagnostic section 3 : seule valeur manquante. La masse salariale ne permet pas de la déduire (05b_charges)")
journaliser("N7", "effectif_impute", "indicateur ajouté (effectif imputé par la règle N7)", int(ventes.effectif_impute.sum()),
            "Permet d'exclure le magasin d'un calcul ou de tester la sensibilité d'une conclusion")

## 9. Règle N8 : ventes datées avant l'ouverture du magasin

**Constat du diagnostic (section 9)** : six magasins ouvrent pendant les trois années (Glasgow, Düsseldorf et Roma en 2019, Marseille et Berlin en 2020, Venezia en 2021). Leur première vente régulière tombe le jour de l'ouverture, mais ils portent aussi 189 lignes datées d'avant cette date, toutes en 2019, pour 1,14 M$ (0,05 % du chiffre d'affaires).

**Règle** : on garde ces lignes et on ajoute l'indicateur `avant_ouverture`. Les supprimer créerait un écart avec le plan stratégique, qui contient ces ventes : l'égalité au cent près entre nos ventes et le plan est ce qui permet de rapprocher les ventes des charges.

**À quoi sert l'indicateur** : à compter correctement les mois d'activité d'un magasin. Un magasin ouvert en octobre 2020 n'a que trois mois d'activité en 2020 : le comparer sur un total annuel à un magasin ouvert toute l'année le ferait passer pour un mauvais magasin. `05_variables` s'appuie sur cet indicateur pour repérer, pour chaque magasin et chaque année, s'il a été ouvert l'année entière.

In [ ]:
# Vrai si la vente est datée avant la date d'ouverture de sa boutique
ventes["avant_ouverture"] = ventes.date < ventes.date_ouverture

journaliser("N8", "avant_ouverture", "indicateur ajouté (vente datée avant l'ouverture de la boutique)",
            ventes.avant_ouverture.sum(),
            "Diagnostic section 9 : ventes gardées (elles font partie des totaux du plan stratégique)")

display(ventes[ventes.avant_ouverture]
        .groupby(["nom_boutique", "date_ouverture"])
        .agg(lignes=("id_vente", "size"), chiffre_affaires=("montant_ventes", "sum"),
             premiere_vente=("date", "min"), derniere_vente=("date", "max"))
        .reset_index().sort_values("date_ouverture"))
print(f"Chiffre d'affaires des ventes avant ouverture : {float(ventes.loc[ventes.avant_ouverture, 'montant_ventes'].sum()):,.2f} $".replace(",", " "),
      f"({float(ventes.loc[ventes.avant_ouverture, 'montant_ventes'].sum()) / float(ventes.montant_ventes.sum()):.2%} du total)")

## 10. Dimensions

Les tables de dimension servent à Power BI et à Excel, qui refont les jointures de leur côté. On leur applique la même règle N1 (et la règle N7 à la dimension des boutiques), pour qu'une marque ou une sous-catégorie s'écrive de la même façon dans la table des ventes et dans sa dimension. Les autres tables (calendrier, contexte par pays et par jour) sont recopiées telles quelles.

In [ ]:
dimensions = {}

# 1. Dimensions réalignées par 02_flocon : boutique, produit, promotion, canal
for chemin_dim in sorted(config.DIM_REALIGNEES.glob("*.parquet")):
    dimensions[chemin_dim.stem] = pd.read_parquet(chemin_dim)

# 2. Calendrier et contexte (jours fériés, sévérité COVID), s'ils existent
for nom in ["dim_date", "contexte_pays_date"]:
    chemin_table = config.INTERIM / f"{nom}.parquet"
    if chemin_table.exists():
        dimensions[nom] = pd.read_parquet(chemin_table)

# 3. Règle N7 : la dimension des boutiques reçoit la même valeur imputée et le même indicateur que la table des ventes
boutique_dim = dimensions["dim_boutique"]
boutique_dim["effectif_impute"] = boutique_dim.id_boutique.isin(effectifs_imputes)
for id_boutique, valeur in effectifs_imputes.items():
    boutique_dim.loc[boutique_dim.id_boutique == id_boutique, "effectif"] = valeur
journaliser("N7", "dim_boutique", "effectif imputé et indicateur effectif_impute ajouté", int(boutique_dim.effectif_impute.sum()),
            "Même valeur que dans la table des ventes")

for nom, table in dimensions.items():
    modifiees = 0
    for col in table.columns:
        if table[col].dtype == object or str(table[col].dtype) in ("str", "string"):
            apres = nettoyer_texte(table[col])
            modifiees += int(((table[col] != apres) & table[col].notna()).sum())
            table[col] = apres
    if modifiees:
        journaliser("N1", nom, "espaces en trop retirés dans les colonnes texte", modifiees,
                    "Même écriture des modalités dans les ventes et dans les dimensions")
    print(f"{nom:<22} {len(table):>6} lignes · {table.shape[1]:>2} colonnes")

## 11. Validation

Avant d'écrire dans `data/clean`, on prouve que le nettoyage a fait ce qui était prévu **et rien d'autre**. Chaque contrôle est enregistré dans un rapport de validation. Si un contrôle échoue, le notebook s'arrête et rien n'est écrit.

In [ ]:
validation = []


def valider(controle, condition, detail=""):
    """Enregistre le résultat d'un contrôle (réussi ou échoué) et l'affiche."""
    resultat = "réussi" if bool(condition) else "ÉCHOUÉ"
    validation.append({"controle": controle, "resultat": resultat, "detail": str(detail)})
    print(f"[{resultat}] {controle} {('· ' + str(detail)) if detail != '' else ''}")


# 1. Aucune ligne ajoutée ni supprimée
valider("Nombre de lignes inchangé", len(ventes) == len(reference), f"{len(ventes)} lignes")
valider("Identifiants de vente identiques et dans le même ordre", (ventes.id_vente.values == reference.id_vente.values).all())
valider("Identifiants de vente uniques", ventes.id_vente.is_unique)

# 2. Mesures : identiques avant et après, LIGNE PAR LIGNE, sauf la quantité remisée (règle N6)
#    sur les seules lignes marquées remise_corrigee.
MESURES = ["quantite_vente", "prix_unitaire", "cout_unitaire", "montant_ventes", "cout_total",
           "quantite_remise", "montant_remise", "quantite_retour", "montant_retour"]
for col in MESURES:
    differe = (ventes[col].astype(float) - reference[col].astype(float)).abs().values > 0.00005
    apres = float(ventes[col].sum())
    if col in COLONNES_N6:
        # Les seules lignes modifiées doivent être celles de la règle N6
        valider(f"Mesure modifiée seulement par N6 : {col}", not (differe & ~ventes.remise_corrigee.values).any(),
                f"{int(differe.sum())} lignes modifiées · total = {apres:,.2f}".replace(",", " "))
    else:
        valider(f"Mesure inchangée : {col}", not differe.any(), f"total = {apres:,.2f}".replace(",", " "))

# 2 bis. Règle N6 : après correction, R5 et R1 sont respectées partout, et les montants sont intacts
valider("N6 : quantité remisée ≤ quantité vendue sur toutes les lignes",
        (ventes.quantite_remise <= ventes.quantite_vente).all())
valider("N6 : montant des ventes = prix × quantité − remise (R1) sur toutes les lignes",
        ((ventes.montant_ventes - (ventes.prix_unitaire * ventes.quantite_vente - ventes.montant_remise)).abs() < 0.01).all())
valider("N6 : chiffre d'affaires total identique à la source",
        abs(float(ventes.montant_ventes.sum()) - float(reference.montant_ventes.sum())) < 0.005,
        f"{float(ventes.montant_ventes.sum()):,.2f} $".replace(",", " "))
# R2 (remise = prix × quantité remisée × taux annoncé) reste vraie partout SAUF sur les lignes N6,
# où la remise conservée dépasse ce que donne le taux annoncé : c'est la limite assumée de la règle.
ecart_r2 = (ventes.montant_remise - ventes.prix_unitaire * ventes.quantite_remise * ventes.taux_remise_annonce).abs() >= 0.01
valider("N6 : la règle R2 n'est en écart que sur les lignes marquées remise_corrigee",
        not (ecart_r2 & ~ventes.remise_corrigee).any(), f"{int(ecart_r2.sum())} lignes en écart")

# 3. Règle N1 : plus d'espace en trop, et les modalités fusionnées sont bien moins nombreuses
valider("Aucun espace en trop dans les colonnes texte",
        all(((ventes[c] == nettoyer_texte(ventes[c])) | ventes[c].isna()).all() for c in colonnes_texte if c in ventes.columns))
valider("Marques : doublons d'écriture fusionnés", ventes.marque.nunique() <= reference.marque.nunique(),
        f"{reference.marque.nunique()} -> {ventes.marque.nunique()} marques")

# 4. Règle N2 : colonnes supprimées
valider("Colonnes sans information absentes de la table propre",
        not any(c in ventes.columns for c in COLONNES_SANS_INFORMATION))

# 5. Règles N3, N4, N6 et N8 : indicateurs présents, sans vide, de type booléen
for col in ["promo_hors_periode", "grande_quantite", "remise_corrigee", "avant_ouverture"]:
    valider(f"Indicateur {col} présent, booléen et sans vide",
            col in ventes.columns and ventes[col].dtype == bool and ventes[col].notna().all(),
            f"{int(ventes[col].sum())} lignes marquées")

# 5 bis. Règle N7 : l'effectif n'a plus de vide, et seules les lignes du magasin imputé ont changé
valider("N7 : effectif sans vide", ventes.effectif.notna().all())
valider("N7 : indicateur effectif_impute présent, booléen et sans vide",
        ventes.effectif_impute.dtype == bool and ventes.effectif_impute.notna().all(),
        f"{int(ventes.effectif_impute.sum())} lignes · {ventes.loc[ventes.effectif_impute, 'id_boutique'].nunique()} magasin")
valider("N7 : effectif inchangé partout où il était connu",
        (ventes.effectif[~ventes.effectif_impute].values == reference.effectif[~ventes.effectif_impute.values].values).all())
valider("N7 : les lignes imputées sont exactement celles dont l'effectif était vide",
        (ventes.effectif_impute.values == reference.effectif.isna().values).all())

# 6. Seul vide toléré : nom_ferie (jour non férié)
VIDES_TOLERES = {"nom_ferie"}
colonnes_avec_vides = set(ventes.columns[ventes.isna().any()])
valider("Aucun vide en dehors de la colonne tolérée (nom_ferie)",
        colonnes_avec_vides <= VIDES_TOLERES, sorted(colonnes_avec_vides))

# 7. Colonnes de la table propre = colonnes d'origine − supprimées + indicateurs
attendues = (set(reference.columns) - set(COLONNES_SANS_INFORMATION)) | {"promo_hors_periode", "grande_quantite", "remise_corrigee", "effectif_impute", "avant_ouverture"}
valider("Colonnes de la table propre conformes", set(ventes.columns) == attendues, f"{ventes.shape[1]} colonnes")

rapport_validation = pd.DataFrame(validation)
echecs = rapport_validation[rapport_validation.resultat != "réussi"]
print(f"\n{len(rapport_validation) - len(echecs)} contrôles réussis sur {len(rapport_validation)}")
assert echecs.empty, "Validation échouée : rien n'est écrit dans data/clean."

## 12. Enregistrement

In [ ]:
# ---------------------------------------------------------------------------
# 1. Journal de nettoyage
# ---------------------------------------------------------------------------
journal_df = pd.DataFrame(journal)
display(journal_df)

# ---------------------------------------------------------------------------
# 2. Écriture dans data/clean
# ---------------------------------------------------------------------------
config.CLEAN.mkdir(parents=True, exist_ok=True)
config.CLEAN_DIMENSIONS.mkdir(parents=True, exist_ok=True)

chemin_fait = config.CLEAN / "fait_ventes.parquet"
ventes.to_parquet(chemin_fait, index=False)

for nom, table in dimensions.items():
    table.to_parquet(config.CLEAN_DIMENSIONS / f"{nom}.parquet", index=False)

# Le journal et le rapport sont en CSV : ils doivent pouvoir être ouverts dans Excel et joints au livrable L3.
# utf-8-sig : Excel affiche correctement les accents.
journal_df.to_csv(config.CLEAN / "journal_nettoyage.csv", index=False, encoding="utf-8-sig")
rapport_validation.to_csv(config.CLEAN / "rapport_validation.csv", index=False, encoding="utf-8-sig")
# Quantité remisée d'origine et corrigée des lignes de la règle N6 (une ligne par vente corrigée)
corrections.to_csv(config.CLEAN / "corrections_remise.csv", index=False, encoding="utf-8-sig")

print("Table propre        :", chemin_fait, f"({len(ventes):,} lignes × {ventes.shape[1]} colonnes)".replace(",", " "))
print("Dimensions          :", config.CLEAN_DIMENSIONS, f"({len(dimensions)} tables)")
print("Journal             :", config.CLEAN / "journal_nettoyage.csv", f"({len(journal_df)} entrées)")
print("Corrections N6      :", config.CLEAN / "corrections_remise.csv", f"({len(corrections)} lignes)")
print("Rapport validation  :", config.CLEAN / "rapport_validation.csv", f"({len(rapport_validation)} contrôles)")

**Suite** : `05_variables` lit `data/clean/fait_ventes.parquet` et calcule les variables d'analyse : chiffre d'affaires brut et net, coût brut, coût du retour et coût net, marges en valeur, taux de remise effectif, taux de retour.